# Demo de inferencia: GPT-2, RoPE, LoRA y Full Fine-Tuning

Este notebook ejecuta las mismas frases con hasta cuatro variantes:

1. **GPT-2 original**: posiciones aprendidas y pesos oficiales.
2. **RoPE original**: pesos oficiales, descartando `wpe` (baseline RoPE sin adaptar).
3. **RoPE + LoRA**: adaptadores entrenados sobre Q/V.
4. **RoPE + Full fine-tuning**: checkpoint completo, si existe en `gpt2_weights/full/`.

Los resultados son demostrativos: RoPE original no fue entrenado con ese esquema posicional. El notebook detecta el checkpoint LoRA más reciente y, cuando exista, el full checkpoint más reciente.

## 1. Entorno y rutas

Ejecuta el notebook desde la raíz del repositorio. El modelo se cargará en GPU si CUDA está disponible.

In [9]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "minigpt").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import tiktoken
import minigpt as mg
from minigpt.lora import inject_lora, load_adapter_state_dict

WEIGHTS_DIR = PROJECT_ROOT / "gpt2_weights"
LORA_RUNS = WEIGHTS_DIR / "lora"
FULL_RUNS = WEIGHTS_DIR / "full"
LORA_CHECKPOINTS = sorted(LORA_RUNS.glob("rope-gpt2-*/best.pt"), key=lambda p: p.stat().st_mtime)
FULL_CHECKPOINTS = sorted(FULL_RUNS.glob("**/best.pt"), key=lambda p: p.stat().st_mtime)
LORA_CHECKPOINT = LORA_CHECKPOINTS[-1] if LORA_CHECKPOINTS else None
FULL_CHECKPOINT = FULL_CHECKPOINTS[-1] if FULL_CHECKPOINTS else None

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = tiktoken.get_encoding("gpt2")
print(f"minigpt {mg.__version__} | torch {torch.__version__} | dispositivo: {device}")
if device == "cuda":
    print(torch.cuda.get_device_name(0))
print("LoRA:", LORA_CHECKPOINT or "no encontrado")
print("Full FT:", FULL_CHECKPOINT or "no encontrado; se omitirá hasta entrenarlo")

minigpt 1.0.0 | torch 2.14.1+cpu | dispositivo: cpu
LoRA: c:\Users\cavir.ROG-CARLOS\Desktop\PLN-Practica\Practica1_PLN1\gpt2_weights\lora\rope-gpt2-20261018-110844\best.pt
Full FT: c:\Users\cavir.ROG-CARLOS\Desktop\PLN-Practica\Practica1_PLN1\gpt2_weights\full\rope-gpt2-20261018-115135\best.pt


## 2. Cargar las variantes

Los adaptadores se cargan sobre una instancia nueva del backbone RoPE. El checkpoint full contiene el `state_dict` completo. Si aún no existe un checkpoint full, se informará y el resto del demo continuará.

In [10]:
models = {}

# 1) GPT-2 original: learned positional embeddings + pesos oficiales.
models["GPT-2 original"] = mg.pretrained_gpt2(
    positional="learned", device=device, weights_dir=WEIGHTS_DIR
)

# 2) RoPE con los mismos pesos oficiales; wpe se descarta deliberadamente.
models["RoPE + pesos originales"] = mg.pretrained_gpt2(
    positional="rope", device=device, weights_dir=WEIGHTS_DIR
)

# 3) RoPE + adaptadores LoRA entrenados.
if LORA_CHECKPOINT is not None:
    adapter_config_path = LORA_CHECKPOINT.parent / "adapter_config.json"
    adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
    rope_lora = mg.pretrained_gpt2(
        positional="rope", device=device, weights_dir=WEIGHTS_DIR
    )
    inject_lora(
        rope_lora,
        tuple(adapter_config["targets"]),
        rank=adapter_config["rank"],
        alpha=adapter_config["alpha"],
        dropout=adapter_config["dropout"],
    )
    lora_data = torch.load(LORA_CHECKPOINT, map_location="cpu", weights_only=False)
    load_adapter_state_dict(rope_lora, lora_data["adapter_state_dict"])
    rope_lora.eval()
    models["RoPE + LoRA"] = rope_lora
    print(f"LoRA cargado (paso {lora_data['step']}, val_loss={lora_data['val_loss']:.4f})")
else:
    print("No hay checkpoint LoRA; ejecuta primero el entrenamiento LoRA.")

# 4) RoPE + full fine-tuning. Se agrega automáticamente cuando exista.
if FULL_CHECKPOINT is not None:
    full_data = torch.load(FULL_CHECKPOINT, map_location="cpu", weights_only=False, mmap=True)
    full_model = mg.GPTModel(mg.GPTConfig(**full_data["model_config"]))
    full_model.load_state_dict(full_data["model_state_dict"])
    full_model.to(device).eval()
    models["RoPE + Full FT"] = full_model
    print(f"Full fine-tuning cargado (paso {full_data['step']}, val_loss={full_data['val_loss']:.4f})")
    del full_data
else:
    print("No hay checkpoint full en gpt2_weights/full/; ese modo aparecerá al entrenarlo.")

for name, model in models.items():
    total = mg.count_parameters(model, trainable_only=False)
    trainable = mg.count_parameters(model, trainable_only=True)
    print(f"{name:<26} total={mg.format_parameters(total):>14} | entrenables={mg.format_parameters(trainable):>10}")

Cargando pesos de GPT-2 desde c:\Users\cavir.ROG-CARLOS\Desktop\PLN-Practica\Practica1_PLN1\gpt2_weights ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Cargando pesos de GPT-2 desde c:\Users\cavir.ROG-CARLOS\Desktop\PLN-Practica\Practica1_PLN1\gpt2_weights ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Cargando pesos de GPT-2 desde c:\Users\cavir.ROG-CARLOS\Desktop\PLN-Practica\Practica1_PLN1\gpt2_weights ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

LoRA cargado (paso 18400, val_loss=4.7103)
Full fine-tuning cargado (paso 20000, val_loss=3.7728)
GPT-2 original             total=   163.037.184 | entrenables=163.037.184
RoPE + pesos originales    total=   162.250.752 | entrenables=162.250.752
RoPE + LoRA                total=   162.545.664 | entrenables=   294.912
RoPE + Full FT             total=   162.250.752 | entrenables=162.250.752


## 3. Comparación con prompts variados

Se prueban dos fragmentos breves del corpus español, dos continuaciones en español con temas nuevos y dos prompts en inglés como control. Reutilizamos cada prompt en todos los modelos. La decodificación greedy hace la comparación reproducible; el contexto se limita a 512 tokens. Los fragmentos del corpus son pistas de estilo/tema: no se afirma que esos artículos concretos hayan aparecido en los minibatches muestreados.

In [11]:
@torch.inference_mode()
def completar(model, prompt, max_new_tokens=24):
    ids = tokenizer.encode(prompt)
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    context_size = min(512, model.cfg.context_length)
    output = mg.generate(
        model, input_ids, max_new_tokens=max_new_tokens,
        context_size=context_size, temperature=None
    )
    return tokenizer.decode(output[0].tolist())

In [12]:
prompt_cases = [
    ("Fragmento del corpus (empleo)", "Más de 860.000 jóvenes menores de 25 años abandonaron el mercado laboral"),
    ("Fragmento del corpus (clima)", "El informe de la Agencia Nacional de Océanos y Atmósfera de Estados Unidos"),
    ("Español: tema nuevo", "Las autoridades anunciaron esta mañana que el nuevo plan"),
    ("Español: tema nuevo", "La investigación publicada por la universidad concluye que"),
    ("English control", "The report published this morning said the new policy would"),
    ("English control", "After a difficult season, the team announced that"),
]

for label, prompt in prompt_cases:
    print("\n" + "=" * 100)
    print(f"CASO: {label}\nPROMPT: {prompt}")
    for name, model in models.items():
        print(f"\n[{name}]")
        print(completar(model, prompt, max_new_tokens=32))


CASO: Fragmento del corpus (empleo)
PROMPT: Más de 860.000 jóvenes menores de 25 años abandonaron el mercado laboral

[GPT-2 original]
Más de 860.000 jóvenes menores de 25 años abandonaron el mercado laboral de la vida de la vida de la vida de la vida de la vida de la vida de la vida de la vida

[RoPE + pesos originales]
Más de 860.000 jóvenes menores de 25 años abandonaron el mercado laboral

































[RoPE + LoRA]
Más de 860.000 jóvenes menores de 25 años abandonaron el mercado laboral de de de de, el más de la vida de de de de de de de de de de de de de la de la de de de de

[RoPE + Full FT]
Más de 860.000 jóvenes menores de 25 años abandonaron el mercado laboral de la ciudad de la ciudad de la ciudad de la ciudad de la ciudad de la

CASO: Fragmento del corpus (clima)
PROMPT: El informe de la Agencia Nacional de Océanos y Atmósfera de Estados Unidos

[GPT-2 original]
El informe de la Agencia Nacional de Océanos y Atmósfera de Estados Unidos.

The following is

## Notas

- El GPT-2 original y RoPE original usan pesos oficiales, pero solo el primero conserva `wpe`.
- LoRA usa el checkpoint más reciente encontrado bajo `gpt2_weights/lora/rope-gpt2-*/best.pt`.
- Full FT se carga desde el `best.pt` más reciente bajo `gpt2_weights/full/`. Para habilitar esa comparación, primero ejecuta `python -m minigpt.finetune --mode full ...`.
- Los modos comparan generaciones; no son una evaluación estadística de calidad.